In [1]:
"""
Image-level fluorescence feature extraction — multi-protein, two-isoform
=========================================================================
Server-mode script: all output goes to a log file (and stdout).
No figures are generated. Run with:

    nohup python image_level_analysis.py > analysis.log 2>&1 &
    tail -f analysis.log

Folder structure:
    ROOT_DIR/
        PROTEIN/
            ISOFORM-A/
                img1.tiff
            ISOFORM-B/
                img1.tiff
        control/
            EGFP-NLS/
                img1.tiff
            ATXN1/
                img1.tiff

Features computed (count depends on DOWNSAMPLE_FACTOR — auto-scaled):
    cv, skewness, kurtosis,
    morans_i_lag{k} × len(MORAN_LAGS), morans_decay,
    texture_contrast/correlation × len(TEXTURE_SCALES),
    granularity_{k} (cumulative binary erosion, k=1..GRAN_MAX) × GRAN_MAX
All spatial scales auto-adjust to preserve physical coverage when DOWNSAMPLE_FACTOR changes.

Outputs:
    per_image_features.csv      one row per image
    summary_per_protein.csv     mean ± SD + Cohen's d + p-value per protein × feature
    analysis.log                full console log (when redirected)

Install:
    pip install tifffile numpy scikit-image scipy pandas imagecodecs
"""

import os, sys, glob, logging, time, re
os.environ['PYTHONUNBUFFERED'] = '1'
import tifffile
import numpy as np
from skimage import filters, morphology, transform
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import skew, kurtosis, ttest_ind
from scipy import sparse

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ── USER SETTINGS ─────────────────────────────────────────────────────────────
ROOT_DIR         = './images_qc_pass/'
IMG_EXT          = '*.tif*'
OUT_DIR          = './outputs/'   # CSVs written one level up
DOWNSAMPLE_FACTOR = 2   # ← only value you need to change (1, 2, or 4)
# ── Derived spatial parameters (all in pixels at the downsampled resolution) ──
_DS = DOWNSAMPLE_FACTOR  # shorthand

# Granularity: cumulative binary erosion, k=1..GRAN_MAX (CellProfiler MeasureGranularity)
# granularity_k captures signal in structures with half-width exactly k px.
# Anchor: GRAN_MAX=10 at DS=2; scales with DS to cover the same pixel range.
GRAN_MAX  = max(4, round(10 * 2 / _DS))   # = 10 at DS=2, 20 at DS=1, 5 at DS=4
GRAN_KEYS = list(range(1, GRAN_MAX + 1))

# GLCM texture scales (px at downsampled resolution)
# Base: [2,4,6,10,20,30] px at DS=2; scale proportionally with DS.
TEXTURE_SCALES = sorted(set(max(1, round(s * 2 / _DS)) for s in [2, 4, 6, 10, 20, 30]))

# Moran's I lags (px at downsampled resolution)
# Base: [1,3,5,10,20] px at DS=2; scale proportionally with DS.
MORAN_LAGS = sorted(set(max(1, round(lag * 2 / _DS)) for lag in [1, 3, 5, 10, 20]))
if 1 not in MORAN_LAGS:
    MORAN_LAGS = [1] + MORAN_LAGS

# Foreground min_size (px² at downsampled resolution)
# Base: 50 px² at DS=2; scales with DS.
FG_MIN_SIZE = max(5, round(50 / (_DS / 2) ** 2))

# Condensate min_size (px² at downsampled resolution)
COND_MIN_SIZE = max(2, round(5 / (_DS / 2) ** 2))

# ── ECDF quantile grid ────────────────────────────────────────────────────────
# 50 evenly-spaced quantile points of the foreground intensity distribution.
# Stored per image; used in summarize_features.py to reconstruct per-isoform
# mean ECDFs and compute KS distance + p-value between L and S isoforms.
ECDF_QUANTILES = np.linspace(0.02, 0.98, 50)   # avoids 0/1 edge effects
ECDF_KEYS      = [f'ecdf_q{int(round(q*100)):02d}' for q in ECDF_QUANTILES]

# ── Features in fixed column order ────────────────────────────────────────────
# Total: 3 + (len(MORAN_LAGS)+1) + 2*len(TEXTURE_SCALES) + len(GRAN_KEYS) + 50
FEAT_COLS = (
    ['cv', 'skewness', 'kurtosis'] +
    [f'morans_i_lag{k}' for k in MORAN_LAGS] +
    ['morans_decay'] +
    [f'texture_contrast_s{s}'    for s in TEXTURE_SCALES] +
    [f'texture_correlation_s{s}' for s in TEXTURE_SCALES] +
    [f'granularity_{k}'          for k in GRAN_KEYS] +
    ECDF_KEYS
)

# ── Logging: write to file AND stdout ─────────────────────────────────────────
LOG_FILE = os.path.join(OUT_DIR, 'analysis.log')
os.makedirs(OUT_DIR, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(LOG_FILE, mode='w'),
        logging.StreamHandler(sys.stdout),
    ]
)
log = logging.getLogger()

# Force stdout to flush immediately (important for tail -f)
sys.stdout.reconfigure(line_buffering=True) if hasattr(sys.stdout, 'reconfigure') else None

log.info(f"ROOT_DIR : {ROOT_DIR}")
log.info(f"OUT_DIR  : {OUT_DIR}")
log.info(f"LOG_FILE : {LOG_FILE}")
log.info(f"Features : {len(FEAT_COLS)}")

2026-08-18 19:30:41  ROOT_DIR : ./images_qc_pass/
2026-08-18 19:30:41  OUT_DIR  : ./outputs/
2026-08-18 19:30:41  LOG_FILE : ./outputs/analysis.log
2026-08-18 19:30:41  Features : 81


In [3]:
# ── Granularity spectrum (CellProfiler MeasureGranularity) ───────────────────
def measure_granularity(img_fg, mask, gran_max):
    """
    CellProfiler-faithful granularity spectrum via cumulative binary erosion.

    At each step k, erode the binary mask by one additional pixel (disk(1)),
    measure the fraction of total foreground intensity lost in that step.
    No dilation back — pure erosion granulometry as described by Ravkin.

    granularity_k = 100 * (intensity_under_eroded_{k-1} - intensity_under_eroded_k)
                         / intensity_under_original_mask

    Physical interpretation: granularity_k captures signal in structures
    whose half-width is exactly k pixels (at the downsampled resolution).

    Parameters
    ----------
    img_fg : 2D float32 ndarray
        Foreground-masked intensity image (background pixels = 0).
        Already downsampled and normalized to [0, 1].
    mask : 2D bool ndarray
        Binary foreground mask (same shape as img_fg).
    gran_max : int
        Number of granularity scales (k = 1 .. gran_max).

    Returns
    -------
    dict  {'granularity_1': float, ..., 'granularity_N': float}
        Values are percentages. All keys always present (0.0 if mask
        fully eroded before reaching that scale).
    """
    total = float(img_fg.sum())
    feats = {f'granularity_{k}': 0.0 for k in range(1, gran_max + 1)}
    if total == 0:
        return feats
    eroded    = mask.copy()
    prev_wsum = total
    selem     = morphology.disk(1)
    for k in range(1, gran_max + 1):
        eroded    = morphology.binary_erosion(eroded, selem)
        curr_wsum = float((img_fg * eroded).sum())
        feats[f'granularity_{k}'] = 100.0 * (prev_wsum - curr_wsum) / total
        prev_wsum = curr_wsum
        if eroded.sum() == 0:
            break   # mask fully eroded; remaining keys stay 0.0
    return feats


# ── Moran's I spatial correlogram ────────────────────────────────────────────
def _morans_i_correlogram(img_norm, mask, lags):
    """
    Compute Moran's I at each Chebyshev ring lag within the foreground mask.

    For each lag k, only pixel pairs at Chebyshev distance exactly k are used
    (the outer ring, not all pixels within distance k). This gives a true
    spatial correlogram rather than cumulative autocorrelation.

    Parameters
    ----------
    img_norm : 2D float array, values in [0, 1]
    mask     : 2D bool array, foreground pixels
    lags     : list of int, Chebyshev ring distances

    Returns
    -------
    list of float, Moran's I at each lag (same order as lags)
    """
    ys, xs = np.where(mask)
    n = len(ys)
    if n < 10:
        return [0.0] * len(lags)

    H, W = mask.shape
    flat_idx = np.full((H, W), -1, dtype=np.int32)
    flat_idx[ys, xs] = np.arange(n)

    vals = img_norm[ys, xs]
    z    = vals - vals.mean()
    z2   = float((z * z).sum())

    results = []
    for lag in lags:
        rows, cols = [], []
        for dy in range(-lag, lag + 1):
            for dx in range(-lag, lag + 1):
                if max(abs(dy), abs(dx)) != lag:
                    continue  # only the outer Chebyshev ring
                ny = ys + dy
                nx = xs + dx
                valid = (ny >= 0) & (ny < H) & (nx >= 0) & (nx < W)
                nidx  = np.where(valid,
                                 flat_idx[np.clip(ny, 0, H-1), np.clip(nx, 0, W-1)],
                                 -1)
                has   = valid & (nidx >= 0)
                rows.append(np.where(has)[0])
                cols.append(nidx[has])

        if not rows or not any(len(r) for r in rows):
            results.append(0.0)
            continue

        rows_cat = np.concatenate(rows)
        cols_cat = np.concatenate(cols)
        W_mat = sparse.csr_matrix(
            (np.ones(len(rows_cat)), (rows_cat, cols_cat)), shape=(n, n))
        S0  = float(W_mat.sum())
        num = float(W_mat.dot(z) @ z)
        mi  = (n / S0) * (num / z2) if z2 > 0 and S0 > 0 else 0.0
        results.append(mi)

    return results


# ── Foreground mask ───────────────────────────────────────────────────────────
def make_mask(img_norm, min_size=None):
    if min_size is None:
        min_size = FG_MIN_SIZE
    t    = filters.threshold_triangle(img_norm)
    mask = img_norm > t
    mask = morphology.remove_small_objects(mask, min_size=min_size)
    return mask

# ── Per-image feature extraction ─────────────────────────────────────────────
def extract_features(img_path):
    img_raw = tifffile.imread(img_path)
    if img_raw.ndim > 2:
        img_raw = img_raw[..., 0]
    img_raw = img_raw.astype(np.float32)
    if DOWNSAMPLE_FACTOR > 1:
        img_raw = transform.downscale_local_mean(
            img_raw, (DOWNSAMPLE_FACTOR, DOWNSAMPLE_FACTOR)).astype(np.float32)
    if img_raw.max() == 0:
        return None, 'blank image'

    img_norm = img_raw / img_raw.max()
    mask     = make_mask(img_norm)
    if mask.sum() < 100:
        return None, f'insufficient foreground ({mask.sum()} px)'

    px     = img_norm[mask]
    img_fg = img_norm * mask
    feats  = {}

    feats['cv']       = px.std() / px.mean()
    feats['skewness'] = skew(px)
    feats['kurtosis'] = kurtosis(px)

    # ── Moran's I spatial correlogram ─────────────────────────────────────────
    # Measures spatial autocorrelation of intensity at each lag distance.
    # The rate of decay with lag separates condensate (fast decay, tight clusters)
    # from diffuse (slow decay, large smooth regions).
    # Uses Chebyshev ring at each lag (only pixel pairs at exactly that distance).
    mi_vals = _morans_i_correlogram(img_norm, mask, MORAN_LAGS)
    for lag, mi in zip(MORAN_LAGS, mi_vals):
        feats[f'morans_i_lag{lag}'] = mi
    # morans_decay = lag1 - lag10; lag10 is at index 3 in MORAN_LAGS=[1,3,5,10,20]
    feats['morans_decay'] = mi_vals[0] - mi_vals[MORAN_LAGS.index(10)]

    img_uint8 = (img_norm * 255).astype(np.uint8)
    glcm = graycomatrix(img_uint8, distances=TEXTURE_SCALES,
                        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=256, symmetric=True, normed=True)
    for d_i, s in enumerate(TEXTURE_SCALES):
        for prop in ['contrast', 'correlation']:
            feats[f'texture_{prop}_s{s}'] = graycoprops(glcm, prop)[d_i, :].mean()

    feats.update(measure_granularity(img_fg, mask, GRAN_MAX))

    return feats, None

In [4]:
# ── Discover protein folders ──────────────────────────────────────────────────
protein_dirs = sorted([d for d in glob.glob(os.path.join(ROOT_DIR, '*'))
                       if os.path.isdir(d)])

if not protein_dirs:
    log.error(f"No subdirectories found in {ROOT_DIR}")
    sys.exit(1)

log.info(f"Found {len(protein_dirs)} protein folder(s)")

2026-08-18 19:30:58  Found 101 protein folder(s)


In [5]:
# ── Granularity spectrum (CellProfiler MeasureGranularity) ───────────────────
def measure_granularity(img_fg, mask, gran_max):
    """
    CellProfiler-faithful granularity spectrum via cumulative binary erosion.

    At each step k, erode the binary mask by one additional pixel (disk(1)),
    measure the fraction of total foreground intensity lost in that step.
    No dilation back — pure erosion granulometry as described by Ravkin.

    granularity_k = 100 * (intensity_under_eroded_{k-1} - intensity_under_eroded_k)
                         / intensity_under_original_mask

    Physical interpretation: granularity_k captures signal in structures
    whose half-width is exactly k pixels (at the downsampled resolution).

    Parameters
    ----------
    img_fg : 2D float32 ndarray
        Foreground-masked intensity image (background pixels = 0).
        Already downsampled and normalized to [0, 1].
    mask : 2D bool ndarray
        Binary foreground mask (same shape as img_fg).
    gran_max : int
        Number of granularity scales (k = 1 .. gran_max).

    Returns
    -------
    dict  {'granularity_1': float, ..., 'granularity_N': float}
        Values are percentages. All keys always present (0.0 if mask
        fully eroded before reaching that scale).
    """
    total = float(img_fg.sum())
    feats = {f'granularity_{k}': 0.0 for k in range(1, gran_max + 1)}
    if total == 0:
        return feats
    eroded    = mask.copy()
    prev_wsum = total
    selem     = morphology.disk(1)
    for k in range(1, gran_max + 1):
        eroded    = morphology.binary_erosion(eroded, selem)
        curr_wsum = float((img_fg * eroded).sum())
        feats[f'granularity_{k}'] = 100.0 * (prev_wsum - curr_wsum) / total
        prev_wsum = curr_wsum
        if eroded.sum() == 0:
            break   # mask fully eroded; remaining keys stay 0.0
    return feats


# ── Moran's I spatial correlogram ────────────────────────────────────────────
def _morans_i_correlogram(img_norm, mask, lags):
    """
    Compute Moran's I at each Chebyshev ring lag within the foreground mask.

    For each lag k, only pixel pairs at Chebyshev distance exactly k are used
    (the outer ring, not all pixels within distance k). This gives a true
    spatial correlogram rather than cumulative autocorrelation.

    Parameters
    ----------
    img_norm : 2D float array, values in [0, 1]
    mask     : 2D bool array, foreground pixels
    lags     : list of int, Chebyshev ring distances

    Returns
    -------
    list of float, Moran's I at each lag (same order as lags)
    """
    ys, xs = np.where(mask)
    n = len(ys)
    if n < 10:
        return [0.0] * len(lags)

    H, W = mask.shape
    flat_idx = np.full((H, W), -1, dtype=np.int32)
    flat_idx[ys, xs] = np.arange(n)

    vals = img_norm[ys, xs]
    z    = vals - vals.mean()
    z2   = float((z * z).sum())

    results = []
    for lag in lags:
        rows, cols = [], []
        for dy in range(-lag, lag + 1):
            for dx in range(-lag, lag + 1):
                if max(abs(dy), abs(dx)) != lag:
                    continue  # only the outer Chebyshev ring
                ny = ys + dy
                nx = xs + dx
                valid = (ny >= 0) & (ny < H) & (nx >= 0) & (nx < W)
                nidx  = np.where(valid,
                                 flat_idx[np.clip(ny, 0, H-1), np.clip(nx, 0, W-1)],
                                 -1)
                has   = valid & (nidx >= 0)
                rows.append(np.where(has)[0])
                cols.append(nidx[has])

        if not rows or not any(len(r) for r in rows):
            results.append(0.0)
            continue

        rows_cat = np.concatenate(rows)
        cols_cat = np.concatenate(cols)
        W_mat = sparse.csr_matrix(
            (np.ones(len(rows_cat)), (rows_cat, cols_cat)), shape=(n, n))
        S0  = float(W_mat.sum())
        num = float(W_mat.dot(z) @ z)
        mi  = (n / S0) * (num / z2) if z2 > 0 and S0 > 0 else 0.0
        results.append(mi)

    return results


# ── Foreground mask ───────────────────────────────────────────────────────────
def make_mask(img_norm, min_size=None):
    if min_size is None:
        min_size = FG_MIN_SIZE
    t    = filters.threshold_triangle(img_norm)
    mask = img_norm > t
    mask = morphology.remove_small_objects(mask, min_size=min_size)
    return mask

# ── Per-image feature extraction ─────────────────────────────────────────────
def extract_features(img_path):
    img_raw = tifffile.imread(img_path)
    if img_raw.ndim > 2:
        img_raw = img_raw[..., 0]
    img_raw = img_raw.astype(np.float32)
    if DOWNSAMPLE_FACTOR > 1:
        img_raw = transform.downscale_local_mean(
            img_raw, (DOWNSAMPLE_FACTOR, DOWNSAMPLE_FACTOR)).astype(np.float32)
    if img_raw.max() == 0:
        return None, 'blank image'

    img_norm = img_raw / img_raw.max()
    mask     = make_mask(img_norm)
    if mask.sum() < 100:
        return None, f'insufficient foreground ({mask.sum()} px)'

    px     = img_norm[mask]
    img_fg = img_norm * mask
    feats  = {}

    feats['cv']       = px.std() / px.mean()
    feats['skewness'] = skew(px)
    feats['kurtosis'] = kurtosis(px)

    # ── Moran's I spatial correlogram ─────────────────────────────────────────
    # Measures spatial autocorrelation of intensity at each lag distance.
    # The rate of decay with lag separates condensate (fast decay, tight clusters)
    # from diffuse (slow decay, large smooth regions).
    # Uses Chebyshev ring at each lag (only pixel pairs at exactly that distance).
    mi_vals = _morans_i_correlogram(img_norm, mask, MORAN_LAGS)
    for lag, mi in zip(MORAN_LAGS, mi_vals):
        feats[f'morans_i_lag{lag}'] = mi
    # morans_decay = lag1 - lag10; lag10 is at index 3 in MORAN_LAGS=[1,3,5,10,20]
    feats['morans_decay'] = mi_vals[0] - mi_vals[MORAN_LAGS.index(10)]

    img_uint8 = (img_norm * 255).astype(np.uint8)
    glcm = graycomatrix(img_uint8, distances=TEXTURE_SCALES,
                        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=256, symmetric=True, normed=True)
    for d_i, s in enumerate(TEXTURE_SCALES):
        for prop in ['contrast', 'correlation']:
            feats[f'texture_{prop}_s{s}'] = graycoprops(glcm, prop)[d_i, :].mean()

    feats.update(measure_granularity(img_fg, mask, GRAN_MAX))

    # ── ECDF quantile grid ─────────────────────────────────────────────────
    # 50-point quantile grid of foreground pixel intensities.
    # Captures the full intensity distribution shape including upper tail.
    ecdf_vals = np.quantile(px, ECDF_QUANTILES)
    for key, val in zip(ECDF_KEYS, ecdf_vals):
        feats[key] = float(val)

    return feats, None

# ── Discover protein folders ──────────────────────────────────────────────────
protein_dirs = sorted([d for d in glob.glob(os.path.join(ROOT_DIR, '*'))
                       if os.path.isdir(d)])

if not protein_dirs:
    log.error(f"No subdirectories found in {ROOT_DIR}")
    sys.exit(1)

log.info(f"Found {len(protein_dirs)} protein folder(s)")

2026-08-18 19:31:06  Found 101 protein folder(s)


In [ ]:
# ── Process all images ────────────────────────────────────────────────────────
all_records  = []
t_start      = time.time()
n_total_imgs = 0

for pdir in protein_dirs:
    protein = os.path.basename(pdir)

    # Each subdirectory is one isoform (subfolder name = isoform name)
    isoform_dirs = sorted([d for d in glob.glob(os.path.join(pdir, '*'))
                           if os.path.isdir(d)])
    if not isoform_dirs:
        log.warning(f"  {protein}: no isoform subfolders found, skipping")
        continue

    for idir in isoform_dirs:
        isoform = os.path.basename(idir)
        paths   = sorted(glob.glob(os.path.join(idir, IMG_EXT)))
        if not paths:
            log.warning(f"  {protein}/{isoform}: no images found")
            continue

        n_ok = n_skip = n_err = 0
        t0     = time.time()
        n_imgs = len(paths)
        log.info(f"  {protein}/{isoform}: starting {n_imgs} images")

        for i, p in enumerate(paths, 1):
            fname = os.path.basename(p)
            t_img = time.time()
            try:
                feats, reason = extract_features(p)
            except Exception as e:
                log.warning(f"    [{i}/{n_imgs}] ERROR {fname}: {e}")
                n_err += 1
                continue

            if feats is None:
                log.info(f"    [{i}/{n_imgs}] SKIP  {fname}: {reason}")
                n_skip += 1
                continue

            feats['file']    = fname
            feats['protein'] = protein
            feats['isoform'] = isoform
            _m = re.match(r'(r\d+c\d+)', fname)
            feats['well_id'] = _m.group(1) if _m else 'unknown'
            all_records.append(feats)
            n_ok += 1
            log.info(f"    [{i}/{n_imgs}] OK    {fname}  ({time.time()-t_img:.1f}s)")

        elapsed = time.time() - t0
        n_total_imgs += n_imgs
        log.info(f"  {protein}/{isoform}: done — {n_ok} ok, {n_skip} skipped, {n_err} errors "
                 f"(total {elapsed:.1f}s, {elapsed/max(n_ok,1):.1f}s/img)")

2026-08-18 19:31:30    ABI1/ABI1-L: starting 8 images
2026-08-18 19:31:30      [1/8] OK    r03c07f01p01-ch1sk1fk1fl1.tiff  (0.2s)
2026-08-18 19:31:30      [2/8] OK    r03c07f08p01-ch1sk1fk1fl1.tiff  (0.2s)
2026-08-18 19:31:31      [3/8] OK    r03c07f09p01-ch1sk1fk1fl1.tiff  (0.2s)
2026-08-18 19:31:31      [4/8] OK    r03c07f18p01-ch1sk1fk1fl1.tiff  (0.2s)
2026-08-18 19:31:31      [5/8] OK    r03c07f26p01-ch1sk1fk1fl1.tiff  (0.2s)
2026-08-18 19:31:31      [6/8] OK    r06c11f08p01-ch1sk1fk1fl1.tiff  (0.2s)
2026-08-18 19:31:31      [7/8] OK    r06c11f12p01-ch1sk1fk1fl1.tiff  (0.2s)
2026-08-18 19:31:31      [8/8] OK    r06c11f22p01-ch1sk1fk1fl1.tiff  (0.2s)
2026-08-18 19:31:31    ABI1/ABI1-L: done — 8 ok, 0 skipped, 0 errors (total 1.5s, 0.2s/img)
2026-08-18 19:31:31    ABI1/ABI1-S: starting 14 images
2026-08-18 19:31:32      [1/14] OK    r06c10f02p01-ch1sk1fk1fl1.tiff  (0.2s)
2026-08-18 19:31:32      [2/14] OK    r06c10f03p01-ch1sk1fk1fl1.tiff  (0.2s)
2026-08-18 19:31:32      [3/14] OK   

In [7]:
# ── Save per_image_features.csv ───────────────────────────────────────────────
df_all = pd.DataFrame(all_records)
out_features = os.path.join(OUT_DIR, 'per_image_features.csv')
df_all.to_csv(out_features, index=False)
log.info(f"Saved {out_features}  ({len(df_all)} rows)")

total_elapsed = time.time() - t_start
log.info(f"Done. {len(df_all)} images processed in {total_elapsed/60:.1f} min.")

2026-08-18 21:27:51  Saved ./outputs/per_image_features.csv  (26694 rows)
2026-08-18 21:27:51  Done. 26694 images processed in 116.3 min.
